# Latex-OCR
Is designed for recognizing mathematical equations and formulas from images. It works best on isolated equations and short expressions. It is not suited for complex PDF layouts with tables, long text passages, or mixed content such as annual reports.


In [ ]:
import sys, os, shutil, subprocess
from pathlib import Path

def in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

IS_COLAB = in_colab()
print(f"Miljö: {'Google Colab' if IS_COLAB else 'Lokal'}")


Miljö: Google Colab


In [ ]:
def install_deps():
    """Installerar allt som behövs. Idempotent — hoppar över om redan finns."""
    pkgs = [
        "pix2tex",          # LaTeX OCR-modellen
        "pdf2image",        # PDF → PIL-bilder
        "Pillow",
        "tqdm",
    ]
    pip_cmd = [sys.executable, "-m", "pip", "install", "-q"] + pkgs

    if IS_COLAB:
        # Colab har pip globalt, poppler behöver apt
        subprocess.check_call(pip_cmd)
        subprocess.check_call(
            ["apt-get", "-qq", "install", "-y", "poppler-utils"],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
    else:
        # Lokalt samma pip, men poppler installeras av användaren
        subprocess.check_call(pip_cmd)
        if not shutil.which("pdftoppm"):
            print(
                "\n  poppler-utils hittades inte!\n"
                "Installera med:\n"
                "  Ubuntu/Debian : sudo apt install poppler-utils\n"
                "  macOS         : brew install poppler\n"
                "  Windows       : conda install -c conda-forge poppler\n"
            )
            sys.exit(1)

print("Installerar beroenden …")
install_deps()
print(" Beroenden klara.\n")

Installerar beroenden …
 Beroenden klara.



In [ ]:
from pdf2image import convert_from_path
from PIL import Image
from pix2tex.cli import LatexOCR
from tqdm import tqdm
import json, re, tempfile

/usr/local/lib/python3.12/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.24). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_serializers.py:44: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `dict[str, any]` - serialized value may not be as expected [field_name='noise_params', input_value=UniformParams(noise_type=... 0.058823529411764705)]), input_type=UniformParams])
  v = handler(item, index)
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `dict[str, any]` - serialized value may not be as expected [field_name='noise_params', input_value=UniformParams(noise_type=... 0.058823529411764705)])

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]
    print("Using PDF:", pdf_path)
except ImportError:
    pdf_path = "data/benchmark/flexqube_arsredovisning_2022.pdf"
    print(f"Running locally, using: {pdf_path}")


Saving FlexQube.pdf to FlexQube (1).pdf
Using PDF: FlexQube (1).pdf


In [ ]:
DPI = 300

print(f"Converting PDF to images ({DPI} DPI) …")
pages: list[Image.Image] = convert_from_path(str(pdf_path), dpi=DPI)
print(f"Ready for {len(pages)} pages.\n")


Konverterar PDF till bilder (300 DPI) …
✓ 68 sidor.



In [ ]:
print("Load LaTeX OCR-model …")
model = LatexOCR()
print(" Model loaded.\n")


def crop_equation_regions(
    page_img: Image.Image,
    *,
    full_page: bool = True,
) -> list[Image.Image]:
    """
Returns a list of image regions to run OCR on.

full_page=True → run OCR on the entire page
full_page=False → future extension: segment equations automatically
"""
    if full_page:
      return [page_img]


results: list[dict] = []

print("Run LaTeX OCR per page…")
for i, page_img in enumerate(tqdm(pages, desc="OCR")):
    regions = crop_equation_regions(page_img, full_page=True)
    page_latex: list[str] = []
    for region in regions:
        try:
            latex = model(region)
            page_latex.append(latex)
        except Exception as e:
            page_latex.append(f"% OCR-error: {e}")
    results.append({
        "page": i + 1,
        "latex": page_latex,
    })

Laddar LaTeX OCR-modell (första gången tar det ~1 min) …
download weights v0.0.1 to path /usr/local/lib/python3.12/dist-packages/pix2tex/model/checkpoints


weights.pth: 100%|██████████| 97.4M/97.4M [00:00<00:00, 265Mb/s]
image_resizer.pth: 100%|██████████| 18.5M/18.5M [00:00<00:00, 107Mb/s] 


✓ Modell laddad.

Kör LaTeX OCR per sida …


OCR: 100%|██████████| 68/68 [18:05<00:00, 15.97s/it]


In [ ]:
print("\n" + "=" * 60)
print("RESULTAT")
print("=" * 60)

for r in results:
    print(f"\n─ Sida {r['page']} ──")
    for eq in r["latex"]:
        print(eq)



RESULTAT

── Sida 1 ──
\left.\begin{array}{c}{{\frac{3}{5\pi^{0}}2s^{+}x^{0}}}\\ {{\frac{3-p^{-1}}{8\pi^{0}}2-\sin\theta}}\\ {{\frac{1}{20}\frac{1}{6}\frac{1}{s}}\\ {{\frac{1}{8\pi^{0}}\frac{1}{s}}-\frac{1}{\sqrt{-8\pi^{0}}}{\sqrt{5}}}\\ {{\frac{8}{8\pi^{0}}}\sum^{-}\frac{{\sqrt[{\frac{5}{9}}\sqrt{-80{\sqrt{10}}}}{\sqrt{0{\sqrt{80}}}}}}\end{array}\right.

── Sida 2 ──
\begin{array}{r l}{\mathrm{periat}}&{{}^{1}\qquad\varepsilon_{\mathrm{fier}}\scriptstyle\left[\begin{array}{c c}{{\scriptstyle\quad\scriptstyle\scriptstyle\frac{}{}-}}\\ {\scriptstyle\frac{\scriptstyle\mathrm{fierier}}{}-\ \;\scriptstyle\frac{}{}-\ \;\scriptstyle\exp}{}}\end{array}\right]}\right]\quad=\left[\begin{array}{c c}{{\scriptstyle\quad\scriptstyle{\frac{}{}-}}}&{{}}\\ {{\frac{}{\scriptstyle\equiv-}-}}&{{}}&{{}}\\ {{\frac{\scriptstyle\scriptstyle\equiv-}{\sin\infty}-\ \;\scriptstyle\exp}{\sqrt{}-\ \;\scriptstyle\exp}{}}&{{}}&{{}}&{{}}\\ {{\frac{\scriptstyle\scriptstyle\mathrm{fier}-}{}-1\;-\;-1}-{\sqrt{}{}-\ \;\ 

In [ ]:
import os

stem = os.path.splitext(os.path.basename(pdf_path))[0]
md_path = f'{stem}_latex.md'

with open(md_path, 'w') as f:
    f.write(f'# LaTeX OCR — {os.path.basename(pdf_path)}\n\n')
    for r in results:
        f.write(f"# Page {r['page']}\n\n")
        for eq in r["latex"]:          # ← loopa över listan
            f.write(f"$$\n{eq}\n$$\n\n")


files.download(md_path)
print(f'Saved and downloaded: {md_path}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Sparad och nedladdad: FlexQube (1)_latex.md
